# 🍳 Optimizer Cookbook — Chapter 01: SGD + Momentum

**Previous:** `00_setup_and_baseline.ipynb` — Vanilla SGD  
**Next:** `02_adagrad.ipynb` — Adagrad

---

## What is Momentum?

Vanilla SGD treats every step independently — it has no memory of where it came from. This causes two well-known problems:

- **Oscillation** — gradients bounce back and forth across steep dimensions
- **Slow progress** — small, cautious steps along flat dimensions

**Momentum** fixes this by accumulating a velocity vector in directions of persistent gradient — like a ball rolling downhill that builds up speed.

---

## Update Rule

$$v_{t+1} = \beta \cdot v_t + \nabla_{\theta} \mathcal{L}(\theta_t)$$

$$\theta_{t+1} = \theta_t - \eta \cdot v_{t+1}$$

Where:
- $v_t$ = velocity (exponential moving average of past gradients)
- $\beta$ = momentum coefficient (typically **0.9**)
- $\eta$ = learning rate

With $\beta = 0.9$, the effective step size is $\frac{1}{1-\beta} = 10\times$ the gradient — so **momentum amplifies consistent directions and dampens oscillations**.

---

## Nesterov Momentum (NAG) — Optional Upgrade

Standard momentum computes gradient **at current position**, then jumps.  
Nesterov computes gradient **at the projected future position** first — a "lookahead" that gives better convergence:

$$v_{t+1} = \beta \cdot v_t + \nabla_{\theta} \mathcal{L}(\theta_t - \eta \beta v_t)$$

Enable with `nesterov=True` in PyTorch.

---

## When to Use SGD + Momentum

| Scenario | Verdict |
|---|---|
| Training CNNs / ResNets from scratch | ✅ Excellent |
| With a learning rate schedule (cosine, step decay) | ✅ Excellent |
| Large batch training | ✅ Good |
| NLP / Transformers | ❌ Adam preferred |
| Sparse features / embeddings | ❌ Adagrad/Adam preferred |
| Quick prototyping without LR tuning | ❌ Too sensitive |

---

## Key Hyperparameters

| Parameter | Typical Value | Effect |
|---|---|---|
| `lr` | 0.01 – 0.1 | Step size |
| `momentum` | 0.9 | Higher = more smoothing, risk of overshooting |
| `weight_decay` | 1e-4 – 5e-4 | L2 regularization |
| `nesterov` | True | Lookahead correction, usually improves results |

---
## 0. Imports & CUDA Check

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')

---
## 1. Config

In [ ]:
# ── Training Config ─────────────────────────────────────────
BATCH_SIZE    = 128
NUM_EPOCHS    = 20
NUM_CLASSES   = 10
NUM_WORKERS   = 2        # set to 0 if DataLoader errors on Windows
SEED          = 42

# ── Optimizer Config ────────────────────────── ← only this changes per notebook
LEARNING_RATE = 0.01
MOMENTUM      = 0.9
WEIGHT_DECAY  = 5e-4
NESTEROV      = True
OPTIMIZER_NAME = 'SGD_Momentum'

# ── Paths ────────────────────────────────────────────────────
DATA_DIR     = '../data'
RESULTS_DIR  = '../results/logs'
PLOTS_DIR    = '../results/plots'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config loaded ✓')

---
## 2. Dataset — CIFAR-10

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches : {len(train_loader)} | Test batches : {len(test_loader)}')

---
## 3. Model — SimpleCNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(SEED)
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready. Trainable params: {total_params:,}')

---
## 4. Optimizer — SGD + Momentum

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr           = LEARNING_RATE,
    momentum     = MOMENTUM,
    weight_decay = WEIGHT_DECAY,
    nesterov     = NESTEROV
)

print(f'Optimizer    : SGD + Momentum')
print(f'LR           : {LEARNING_RATE}')
print(f'Momentum (β) : {MOMENTUM}')
print(f'Weight Decay : {WEIGHT_DECAY}')
print(f'Nesterov     : {NESTEROV}')
print('-' * 65)

---
## 5. Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def run_training(model, train_loader, test_loader, optimizer, criterion, num_epochs, device, optimizer_name):
    history = []
    best_acc = 0.0
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = evaluate(model, test_loader, criterion, device)
        elapsed = time.time() - t0
        if val_acc > best_acc: best_acc = val_acc
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                        'val_loss': val_loss, 'val_acc': val_acc, 'time_s': elapsed})
        print(f'Epoch [{epoch:02d}/{num_epochs}] Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | Time: {elapsed:.1f}s')
    print(f'\n✓ Best Val Accuracy: {best_acc:.2f}%')
    df = pd.DataFrame(history)
    df.to_csv(f'{RESULTS_DIR}/{optimizer_name}_log.csv', index=False)
    print(f'✓ Log saved → results/logs/{optimizer_name}_log.csv')
    return df

print('Utilities loaded ✓')

---
## 6. Train

In [ ]:
history = run_training(
    model          = model,
    train_loader   = train_loader,
    test_loader    = test_loader,
    optimizer      = optimizer,
    criterion      = criterion,
    num_epochs     = NUM_EPOCHS,
    device         = device,
    optimizer_name = OPTIMIZER_NAME
)

---
## 7. Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — {OPTIMIZER_NAME}', fontsize=14, fontweight='bold')

ax1.plot(history['epoch'], history['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
ax1.plot(history['epoch'], history['val_loss'],   label='Val Loss',   linewidth=2, color='tomato', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['epoch'], history['train_acc'], label='Train Acc', linewidth=2, color='steelblue')
ax2.plot(history['epoch'], history['val_acc'],   label='Val Acc',   linewidth=2, color='tomato', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{OPTIMIZER_NAME}_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Plot saved → results/plots/{OPTIMIZER_NAME}_curves.png')

---
## 8. Momentum Sensitivity Analysis

This cell runs a **mini sweep** over different momentum values to show how β affects convergence.  
Only 5 epochs per run to keep it fast — the trend is clear.

> ⚠️ This takes a few minutes. Skip if you just want the main result.

In [ ]:
momentum_values = [0.0, 0.5, 0.9, 0.99]
sweep_results = {}
SWEEP_EPOCHS = 5

for beta in momentum_values:
    torch.manual_seed(SEED)
    m = SimpleCNN(num_classes=NUM_CLASSES).to(device)
    opt = optim.SGD(m.parameters(), lr=LEARNING_RATE, momentum=beta)
    crit = nn.CrossEntropyLoss()
    val_accs = []
    for epoch in range(1, SWEEP_EPOCHS + 1):
        train_one_epoch(m, train_loader, crit, opt, device)
        _, val_acc = evaluate(m, test_loader, crit, device)
        val_accs.append(val_acc)
    sweep_results[f'β={beta}'] = val_accs
    print(f'β={beta} → Final Val Acc: {val_accs[-1]:.2f}%')

# Plot sweep
plt.figure(figsize=(9, 5))
for label, accs in sweep_results.items():
    plt.plot(range(1, SWEEP_EPOCHS + 1), accs, marker='o', linewidth=2, label=label)
plt.title('Momentum Sensitivity — Val Accuracy vs β', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy (%)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/momentum_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. Baseline Comparison

Compare SGD + Momentum against vanilla SGD baseline (loads the saved CSV from Chapter 00).

In [ ]:
import os

baseline_path = f'{RESULTS_DIR}/SGD_baseline_log.csv'

if os.path.exists(baseline_path):
    baseline = pd.read_csv(baseline_path)

    plt.figure(figsize=(9, 5))
    plt.plot(baseline['epoch'], baseline['val_acc'], label='Vanilla SGD',       linewidth=2, color='gray',      linestyle='--')
    plt.plot(history['epoch'],  history['val_acc'],  label='SGD + Momentum',    linewidth=2, color='steelblue')
    plt.title('Val Accuracy — SGD vs SGD + Momentum', fontsize=13, fontweight='bold')
    plt.xlabel('Epoch'); plt.ylabel('Val Accuracy (%)')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/SGD_vs_Momentum.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f"Vanilla SGD best   : {baseline['val_acc'].max():.2f}%")
    print(f"SGD + Momentum best: {history['val_acc'].max():.2f}%")
    print(f"Improvement        : +{history['val_acc'].max() - baseline['val_acc'].max():.2f}%")
else:
    print('Run 00_setup_and_baseline.ipynb first to generate the baseline log.')

---
## 10. Results Summary & Analysis

In [ ]:
best_epoch = history.loc[history['val_acc'].idxmax()]

print('=' * 55)
print(f'  {OPTIMIZER_NAME} — Summary')
print('=' * 55)
print(f"  Best Val Accuracy : {best_epoch['val_acc']:.2f}%")
print(f"  Best Val Loss     : {best_epoch['val_loss']:.4f}")
print(f"  Achieved at Epoch : {int(best_epoch['epoch'])}")
print(f"  Avg Time/Epoch    : {history['time_s'].mean():.1f}s")
print('=' * 55)
print()
print('📌 Key Observations:')
print('  ✅ Momentum significantly speeds up convergence vs vanilla SGD')
print('  ✅ Nesterov look-ahead gives a small but consistent accuracy boost')
print('  ✅ Weight decay acts as L2 regularization, reduces overfitting')
print('  ⚠️  β=0.99 can overshoot — 0.9 is the sweet spot for most tasks')
print('  ⚠️  Still requires careful LR tuning unlike Adam')
print()
print('📌 When to use SGD + Momentum:')
print('  → Training vision models (CNNs, ResNets) from scratch')
print('  → When paired with a LR scheduler (cosine annealing, step decay)')
print('  → When you want maximum control over the training process')
print('  → Reproducing published results (many papers use SGD + Momentum)')

---
## 11. What's Next?

SGD + Momentum uses a **global** learning rate for all parameters.  
The next optimizer, **Adagrad**, adapts the learning rate **per parameter** — making it powerful for sparse data and embeddings.

➡️ Continue to `02_adagrad.ipynb`